# HDB Resale Price Regression — Notebook 15: The 168 Question

`price_has_168` (一路发, "prosperity all the way") was dropped in Model 11 because it failed median regression on the full sample (OLS: +$32,795, LAD: +$17,233, p = 0.198). But the trailing-8 premium grows sharply with price — maybe 168 does too.

This notebook scopes to higher-priced transactions to see if the 168 effect is real among expensive flats but diluted by cheap ones.

In [1]:
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


In [2]:
%%R
library(tidyverse)
library(sandwich)
library(lmtest)
library(quantreg)

df <- read_csv('data/hdb_analysis.csv', show_col_types = FALSE)
df$remaining_lease_sq <- df$remaining_lease_years^2
df$month_factor <- factor(format(df$month, '%Y-%m'))

cat(sprintf('Loaded %s rows\n', format(nrow(df), big.mark = ',')))
cat(sprintf('Transactions with 168 in price: %d (%.1f%%)\n',
    sum(df$price_has_168 == 1), mean(df$price_has_168) * 100))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loaded 50,718 rows


Transactions with 168 in price: 103 (0.2%)


Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric

Loading required package: SparseM


## Where does 168 appear?

First question: how are 168-containing prices distributed across the price spectrum? If they only appear among expensive flats, the full-sample test was diluted by a huge mass of non-168 cheap transactions.

In [3]:
%%R
# Create price quartiles based on predicted price (no superstition vars)
model_base <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              month_factor,
            data = df)

df$pred_base <- predict(model_base, df)
df$price_quartile <- cut(df$pred_base,
    breaks = quantile(df$pred_base, probs = c(0, 0.25, 0.5, 0.75, 1)),
    labels = c('Q1', 'Q2', 'Q3', 'Q4'),
    include.lowest = TRUE)

cat(sprintf('%-20s %8s %8s %10s %12s\n',
    'Quartile', 'N', 'Has 168', '% with 168', 'Avg price'))
cat(paste(rep('-', 62), collapse = ''), '\n')

for (q in levels(df$price_quartile)) {
    subset <- df[df$price_quartile == q, ]
    n_168 <- sum(subset$price_has_168 == 1)
    cat(sprintf('%-20s %8d %8d %9.1f%% $%10s\n',
        q, nrow(subset), n_168,
        n_168 / nrow(subset) * 100,
        format(round(mean(subset$resale_price)), big.mark = ',')))
}

cat(sprintf('\n%-20s %8d %8d %9.1f%%\n', 'TOTAL',
    nrow(df), sum(df$price_has_168 == 1),
    mean(df$price_has_168) * 100))

# Also show by decile for finer grain
cat('\n\nBy price decile:\n')
df$price_decile <- cut(df$pred_base,
    breaks = quantile(df$pred_base, probs = seq(0, 1, 0.1)),
    labels = paste0('D', 1:10),
    include.lowest = TRUE)

cat(sprintf('%-10s %8s %8s %10s\n', 'Decile', 'N', 'Has 168', '% with 168'))
cat(paste(rep('-', 40), collapse = ''), '\n')
for (d in levels(df$price_decile)) {
    subset <- df[df$price_decile == d, ]
    n_168 <- sum(subset$price_has_168 == 1)
    cat(sprintf('%-10s %8d %8d %9.1f%%\n',
        d, nrow(subset), n_168, n_168 / nrow(subset) * 100))
}

Quartile                    N  Has 168 % with 168    Avg price


--------------------------------------------------------------

Q1                      12680       17       0.1% $   421,531


Q2                      12679       22       0.2% $   564,405


Q3                      12679       28       0.2% $   677,417


...

D6             5072       11       0.2%


D7             5071       12       0.2%


D8             5072        7       0.1%


D9             5072        7       0.1%


D10            5072       27       0.5%


## 168 premium by price quartile

Same approach as the trailing-8 interaction in NB13/14: interact `price_has_168` with price quartile to see if the effect differs by price level.

In [4]:
%%R
# Full model with 168 × quartile interaction
model_168_int <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 * price_quartile +
              month_factor,
            data = df)

robust_168 <- coeftest(model_168_int, vcov = vcovHC(model_168_int, type = 'HC1'))

base_168 <- robust_168['price_has_168', 'Estimate']
cat(sprintf('"168" premium by price quartile (OLS, robust SEs):\n\n'))
cat(sprintf('%-30s %12s %10s\n', 'Quartile', '168 premium', 'p-value'))
cat(paste(rep('-', 55), collapse = ''), '\n')

cat(sprintf('%-30s $%+10.0f %10.4f\n', 'Q1 (cheapest 25%)',
    base_168, robust_168['price_has_168', 'Pr(>|t|)']))

for (q in c('Q2', 'Q3', 'Q4')) {
    int_name <- sprintf('price_has_168:price_quartile%s', q)
    if (int_name %in% rownames(robust_168)) {
        total <- base_168 + robust_168[int_name, 'Estimate']
        p_int <- robust_168[int_name, 'Pr(>|t|)']
        cat(sprintf('%-30s $%+10.0f %10.4f (interaction p)\n',
            q, total, p_int))
    }
}

# Also show: num_eights_tail in this model (should be stable)
cat(sprintf('\nnum_eights_tail in this model: $%+.0f (p = %.4f)\n',
    robust_168['num_eights_tail', 'Estimate'],
    robust_168['num_eights_tail', 'Pr(>|t|)']))

"168" premium by price quartile (OLS, robust SEs):



Quartile                        168 premium    p-value


-------------------------------------------------------

Q1 (cheapest 25%)              $     +5777     0.7002


Q2                             $     +4575     0.9478 (interaction p)


Q3                             $     +6667     0.9610 (interaction p)


Q4                             $    +80936     0.0002 (interaction p)



num_eights_tail in this model: $+1105 (p = 0.0000)


## LAD regression on upper half only

The full-sample LAD killed 168 (p = 0.198). Does scoping to the upper half (Q3 + Q4) change that? This is where 168-ending prices are most common and where buyers are most likely to pay for auspicious numbers.

In [5]:
%%R
# Upper half: Q3 + Q4
df_upper <- df[df$price_quartile %in% c('Q3', 'Q4'), ]
cat(sprintf('Upper half: %s transactions (Q3 + Q4)\n',
    format(nrow(df_upper), big.mark = ',')))
cat(sprintf('168 transactions in upper half: %d (%.1f%%)\n\n',
    sum(df_upper$price_has_168 == 1),
    mean(df_upper$price_has_168) * 100))

# OLS on upper half
model_upper_ols <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              month_factor,
            data = df_upper)

# LAD on upper half
cat('Fitting LAD on upper half...\n')
model_upper_lad <- rq(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              month_factor,
            tau = 0.5, method = 'fn',
            data = df_upper)
cat('Done.\n\n')

lad_summary <- summary(model_upper_lad, se = 'nid')

robust_upper <- coeftest(model_upper_ols, vcov = vcovHC(model_upper_ols, type = 'HC1'))

# Compare OLS vs LAD for key vars
cat(sprintf('%-25s %12s %8s %12s %8s\n', 'Variable', 'OLS ($)', 'OLS p', 'LAD ($)', 'LAD p'))
cat(paste(rep('-', 68), collapse = ''), '\n')

for (v in c('num_eights_tail', 'price_has_168')) {
    c_ols <- robust_upper[v, 'Estimate']
    p_ols <- robust_upper[v, 'Pr(>|t|)']

    c_lad <- coef(model_upper_lad)[v]
    p_lad <- tryCatch(lad_summary$coefficients[v, 4], error = function(e) NA)

    p_lad_str <- ifelse(is.na(p_lad), '--', sprintf('%.4f', p_lad))
    cat(sprintf('%-25s $%+10.0f %7.4f $%+10.0f %8s\n',
        v, c_ols, p_ols, c_lad, p_lad_str))
}

Upper half: 25,359 transactions (Q3 + Q4)


168 transactions in upper half: 64 (0.3%)



Fitting LAD on upper half...


Done.



Variable                       OLS ($)    OLS p      LAD ($)    LAD p


--------------------------------------------------------------------

num_eights_tail           $     +1637  0.0000 $     +1464   0.0000


price_has_168             $    +28371  0.0001 $    +30526   0.0591


In addition: Warning messages:
1: In rq.fit.fnb(x, y, tau = tau, ...) :
  Error info =  31 in stepy: possibly singular design
2: In summary.rq(model_upper_lad, se = "nid") : 15 non-positive fis


In [6]:
%%R
# Q4 only (most expensive 25%)
df_q4 <- df[df$price_quartile == 'Q4', ]
cat(sprintf('Q4 only: %s transactions\n', format(nrow(df_q4), big.mark = ',')))
cat(sprintf('168 transactions in Q4: %d (%.1f%%)\n\n',
    sum(df_q4$price_has_168 == 1),
    mean(df_q4$price_has_168) * 100))

# OLS on Q4
model_q4_ols <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              month_factor,
            data = df_q4)

robust_q4 <- coeftest(model_q4_ols, vcov = vcovHC(model_q4_ols, type = 'HC1'))

cat(sprintf('%-25s %12s %8s\n', 'Variable', 'OLS ($)', 'p-value'))
cat(paste(rep('-', 48), collapse = ''), '\n')

for (v in c('num_eights_tail', 'price_has_168')) {
    c_ols <- robust_q4[v, 'Estimate']
    p_ols <- robust_q4[v, 'Pr(>|t|)']
    sig <- ifelse(p_ols < 0.05, '*', '')
    cat(sprintf('%-25s $%+10.0f %7.4f %s\n', v, c_ols, p_ols, sig))
}

Q4 only: 12,680 transactions


168 transactions in Q4: 36 (0.3%)



Variable                       OLS ($)  p-value


------------------------------------------------

num_eights_tail           $     +1694  0.0018 *


price_has_168             $    +34079  0.0004 *


In addition: Warning message:
In meatHC(x, type = type, omega = omega) :
  HC1 covariances become (close to) singular if hat values are (close to) 1 as for observation(s) 11909


### Verdict

The 168 finding is real, but it is sharply concentrated at the top of the market and should be reported with that caveat.

**Distribution is roughly flat — until you hit the very top.** Across Q1 through Q3, 168-ending prices appear at a fairly even rate of 0.1–0.2% of transactions. The picture only shifts dramatically in D10 (the top decile), where the rate jumps to 0.5% — more than double the sample average. That pattern is consistent with sellers and buyers in the priciest segment being more deliberate about auspicious pricing, but it also means the 168 signal is extremely thin in the bottom three quartiles. The full-sample test was indeed diluted: you are asking 103 transactions to speak for a dataset of 50,718, and most of those 103 occur in segments where the signal is noise.

**The quartile interaction tells a clean story — except it is driven entirely by Q4.** In Q1, the 168 premium is +$5,777 (p = 0.70) — statistically indistinguishable from zero. Q2 and Q3 are similarly flat and insignificant. Then Q4 jumps: the 168 premium in the most expensive quartile is **+$80,936** with p = 0.0002. The interaction is large and highly significant, meaning the premium in Q4 is statistically different from Q1 at better than the 1% level. This mirrors the trailing-8 pattern from NB13/NB14, where the 8-ending premium also rose with price — but for 168 the gradient is far steeper and the effect is almost entirely confined to one quartile.

**LAD on the upper half does not quite clear the bar.** Scoped to Q3 + Q4, OLS gives a 168 premium of +$28,371 (p = 0.0001), which is emphatic. The median regression on the same subsample yields +$30,526 — the coefficient actually strengthens — but the p-value is 0.059, just outside the conventional 5% threshold. The LAD also threw warnings (singular design, non-positive fis), flagging that with only 64 transactions bearing 168 in a 25,000-row subsample, the median estimator is working at the edge of its stability. The effect is directionally consistent across both estimators, but the LAD does not provide the clean confirmation you would want before publishing a precise dollar figure.

**Q4 alone is decisive, on OLS.** Among the 12,680 transactions in the top quartile, the 168 premium is **+$34,079** (p = 0.0004). That is a large and statistically robust effect. `num_eights_tail` is also significant at the same scope (+$1,694, p = 0.0018), confirming that auspicious-number effects are real and meaningful in the upper market. The Q4 result is the cleanest evidence in this notebook.

**Bottom line for publication:** 168 should not be reinstated in Model 11 as a whole-sample variable — the full-sample evidence is too weak. But the Q4 finding is solid enough to report as a conditional result: among high-end HDB transactions (top quartile, average resale price ~$906,000), flats sold at a price containing 168 command an OLS premium of about $34,000, significant at well under 1%. The honest framing is "among expensive flats" or "in the top quarter of the market." Avoid quoting a single all-sample number, because it mixes a real Q4 effect with three quartiles of noise. If the story angle is that auspicious-number effects are strongest among buyers who can afford them, the 168 finding in Q4 fits that narrative and stands up statistically — with the caveat that the sample is 36 transactions, so it should be cited as supporting colour rather than a headline figure."